In [19]:
import pandas as pd
import numpy as np

In [20]:
review_path = r"C:\Users\rnu43\MyFiles\Documents\Python\PROJECTS\FRD_System\All_Beauty.jsonl\All_Beauty.jsonl"
meta_path = r"C:\Users\rnu43\MyFiles\Documents\Python\PROJECTS\FRD_System\meta_All_Beauty.jsonl\meta_All_Beauty.jsonl"

# 1. Load reviews (behavioral and text branch)
reviews_df = pd.read_json(review_path, lines=True, nrows=5000)

# 2. Load metadata (item-context branch)
meta_df = pd.read_json(meta_path, lines=True, nrows=5000)

# 3. Merge on parent_asin
merged_df = pd.merge(
    reviews_df, 
    meta_df[['parent_asin', 'price', 'average_rating', 'rating_number', 'store']], 
    on='parent_asin', 
    how='left'
)

print("--- Multi-Branch Dataset Ready ---")
print(f"Merged shape: {merged_df.shape}")
print(merged_df.head(2))

--- Multi-Branch Dataset Ready ---
Merged shape: (5000, 14)
   rating                                      title  \
0       5  Such a lovely scent but not overpowering.   
1       4     Works great but smells a little weird.   

                                                text images        asin  \
0  This spray is really nice. It smells really go...     []  B00YQ6X8EO   
1  This product does what I need it to do, I just...     []  B081TJ8YS3   

  parent_asin                       user_id               timestamp  \
0  B00YQ6X8EO  AGKHLEW2SOWHNMFQIJGBECAF7INQ 2020-05-05 14:08:48.923   
1  B081TJ8YS3  AGKHLEW2SOWHNMFQIJGBECAF7INQ 2020-05-04 18:10:55.070   

   helpful_vote  verified_purchase  price  average_rating  rating_number store  
0             0               True    NaN             NaN            NaN   NaN  
1             1               True    NaN             NaN            NaN   NaN  


In [34]:
print(len(merged_df))
print(len(reviews_df))
print(merged_df.shape)
print(merged_df.info())

5000
5000
(5000, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   rating             5000 non-null   int64         
 1   title              5000 non-null   object        
 2   text               5000 non-null   object        
 3   images             5000 non-null   object        
 4   asin               5000 non-null   object        
 5   parent_asin        5000 non-null   object        
 6   user_id            5000 non-null   object        
 7   timestamp          5000 non-null   datetime64[ns]
 8   helpful_vote       5000 non-null   int64         
 9   verified_purchase  5000 non-null   bool          
 10  price              100 non-null    float64       
 11  average_rating     286 non-null    float64       
 12  rating_number      286 non-null    float64       
 13  store              268 non-null    object 

In [31]:
merged_df.isna().sum()

rating                  0
title                   0
text                    0
images                  0
asin                    0
parent_asin             0
user_id                 0
timestamp               0
helpful_vote            0
verified_purchase       0
price                4900
average_rating       4714
rating_number        4714
store                4732
char_count              0
word_count              0
clean_text              0
avg_word_length         0
uppercase_ratio         0
exclamation_count       0
dtype: int64

In [23]:
merged_df['user_id'].value_counts().value_counts().head()

count
1    1948
2     321
3     100
4      33
5      12
Name: count, dtype: int64

In [35]:
merged_df['clean_text'] = merged_df['text'].fillna('').astype(str)

merged_df['char_count'] = merged_df['clean_text'].str.len().astype(float)
merged_df['word_count'] = merged_df['clean_text'].str.split().str.len().astype(float)
merged_df['avg_word_length'] = merged_df['char_count'] / (merged_df['word_count'] + 1e-5)
merged_df['uppercase_ratio'] = merged_df['clean_text'].fillna('').apply(
    lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1e-5)
)
merged_df['exclamation_count'] = merged_df['clean_text'].fillna('').apply(lambda x: x.count('!'))

In [37]:
merged_df['is_extreme_rating'] = merged_df['rating'].isin([1, 5]).astype(int)
merged_df['verified_purchase'] = merged_df['verified_purchase'].astype(int)
user_counts = merged_df['user_id'].map(merged_df['user_id'].value_counts())
merged_df['is_cold_start_user'] = (user_counts == 1).astype(int)

In [39]:
merged_df['price_is_missing'] = merged_df['price'].isna().astype(int)
merged_df['store_is_missing'] = merged_df['store'].isna().astype(int)
merged_df['price_cleaned'] = merged_df['price'].fillna(0).astype(float)
print(merged_df[['price', 'price_is_missing', 'price_cleaned']].head(3))

   price  price_is_missing  price_cleaned
0    NaN                 1            0.0
1    NaN                 1            0.0
2    NaN                 1            0.0


In [40]:
print(merged_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   rating              5000 non-null   int64         
 1   title               5000 non-null   object        
 2   text                5000 non-null   object        
 3   images              5000 non-null   object        
 4   asin                5000 non-null   object        
 5   parent_asin         5000 non-null   object        
 6   user_id             5000 non-null   object        
 7   timestamp           5000 non-null   datetime64[ns]
 8   helpful_vote        5000 non-null   int64         
 9   verified_purchase   5000 non-null   int64         
 10  price               100 non-null    float64       
 11  average_rating      286 non-null    float64       
 12  rating_number       286 non-null    float64       
 13  store               268 non-null    object      

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
text_matrix = tfidf.fit_transform(merged_df['clean_text'])

In [42]:
from scipy.sparse import hstack

# Select numerical and behavioral features to combine with our text matrix
feature_cols = [
    'char_count', 'word_count', 'avg_word_length', 
    'uppercase_ratio', 'exclamation_count', 
    'is_extreme_rating', 'verified_purchase', 
    'is_cold_start_user', 'price_is_missing', 
    'store_is_missing', 'price_cleaned'
]

# Extract numerical values and combine efficiently using scipy's hstack
X_num = merged_df[feature_cols].values
X_combined = hstack([text_matrix, X_num])

print("Combined feature matrix shape:", X_combined.shape)

Combined feature matrix shape: (5000, 5011)


In [48]:
import numpy as np

# Condition 1: Extreme rating + Unverified + Zero helpful votes
cond1 = (merged_df['is_extreme_rating'] == 1) & (merged_df['verified_purchase'] == 0) & (merged_df['helpful_vote'] == 0)

# Condition 2: Extreme rating from a brand new/cold-start user with very short text
cond2 = (merged_df['is_extreme_rating'] == 1) & (merged_df['is_cold_start_user'] == 1) & (merged_df['char_count'] < 40)

# Condition 3: Suspiciously repetitive or low-effort formatting (extreme rating + low word count)
cond3 = (merged_df['is_extreme_rating'] == 1) & (merged_df['word_count'] < 5)

# Multi-signal scoring: flag as 1 if multiple suspicious markers align
score = cond1.astype(int) + cond2.astype(int) + cond3.astype(int)
y = (score >= 2).astype(int)

print("Target Class Distribution:")
print(y.value_counts())
print(f"\nPositive Class Ratio: {y.mean() * 100:.2f}%")

Target Class Distribution:
0    4806
1     194
Name: count, dtype: int64

Positive Class Ratio: 3.88%


In [ ]:
from sklearn.linear_model import LogisticRegression

# Initialize model with class weighting to handle the heavy skew
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train, y_train)